<a href="https://colab.research.google.com/github/GitTanish/RAG/blob/master/RagAPP27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import langchain
print(langchain.__version__)

1.2.15


In [30]:
from google.colab import drive
drive.mount('/content/drive')

from getpass import getpass

groq_key = getpass("Enter GROQ_API_KEY: ")
langchain_key = getpass("Enter LANGCHAIN_API_KEY: ")

with open("/content/drive/MyDrive/.env", "w") as f:
    f.write(f"GROQ_API_KEY={groq_key}\n")
    f.write(f"LANGCHAIN_API_KEY={langchain_key}\n")
    f.write("LANGCHAIN_TRACING_V2=true\n")
    f.write("LANGCHAIN_PROJECT=Project chain\n")

Mounted at /content/drive
Enter GROQ_API_KEY: ··········
Enter LANGCHAIN_API_KEY: ··········


In [31]:
!pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/.env")

# verify
print(os.getenv("GROQ_API_KEY")[:5])
print(os.getenv("LANGCHAIN_API_KEY")[:5])
print(os.getenv("LANGCHAIN_PROJECT"))

gsk_6
lsv2_
Project chain


In [25]:
! pip install -qU langchain-groq

In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")
llm_response = llm.invoke('Tell me a joke')
llm_response

AIMessage(content="Why did the scarecrow become a successful motivational speaker?\n\nBecause he was outstanding in his field—and always knew how to lift people's spirits!", additional_kwargs={'reasoning_content': "The user just wants a joke. Provide a joke. Should be appropriate. Let's give a light-hearted joke."}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 75, 'total_tokens': 134, 'completion_time': 0.126586762, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002755612, 'prompt_tokens_details': None, 'queue_time': 0.004631433, 'total_time': 0.129342374}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dc4ca88fa4', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d5e28-220f-7ac2-a2be-2b5eea2ffa91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 59, 'total_tokens': 134, 'output_token_details': {'reasoning

#### Parsing output


In [20]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

"Why did the scarecrow become a successful motivational speaker?\n\nBecause he was outstanding in his field—and always knew how to lift people's spirits!"

#### Simple Chain

In [22]:
chain = llm | output_parser
chain.invoke("How make a scrambled egg")

'## Classic Fluffy Scrambled Eggs  \n*(Serves 1–2 people)*  \n\n| Ingredient | Quantity |\n|------------|----------|\n| Large eggs | 2‑4 (depending on appetite) |\n| Milk, cream, or water | 1\u202f–\u202f2\u202fTbsp (optional, for extra fluff) |\n| Salt | a pinch (to taste) |\n| Freshly ground black pepper | a pinch (to taste) |\n| Butter or oil | 1\u202f–\u202f1½\u202ftsp |\n| Optional add‑ins | shredded cheese, chopped herbs, diced veggies, cooked bacon/ham, etc. |\n\n### Equipment\n- Small non‑stick skillet or frying pan (≈\u202f8\u202fin/20\u202fcm)\n- Silicone or rubber spatula\n- Mixing bowl\n- Fork or small whisk\n- Plate and serving fork\n\n---\n\n## Step‑by‑Step Method\n\n1. **Crack & Beat**  \n   - Crack the eggs into a bowl.  \n   - Add the milk/cream/water (if using) and a pinch of salt.  \n   - Beat with a fork or whisk until the mixture is uniform and a little frothy—about **15–20 seconds**. The more air you incorporate, the fluffier the final eggs.\n\n2. **Pre‑heat the P

In [24]:
from typing import List
from pydantic import BaseModel, Field

class WorkOutPlanner(BaseModel):
  routine: str = Field(description = "When does the speaker wakes and sleep")
  shoulder_exercise: str = Field(description="What are his shoulder specific exercise")
  diet: str = Field(description= "What food he eats for his diet")

transcription = """
Hey It's me XYZ, My day starts at waking up on 7 am , I start my day with scrambled eggs, milk and bread.
Then I work on my various project, at 3 pm. I hit the gym for workout. For chest I do bench press, I can bench a small house I feel.
For shoulders I do Shoulder press and Lateral Raises feels great tho. Like wise I divide the week different muscle groups.

I normally eat chicken breast with some butter , Mackerel and sometimes lobster.
I feel alive and great.

"""

structured_llm = llm.with_structured_output(WorkOutPlanner)
output = structured_llm.invoke(transcription)
output

WorkOutPlanner(routine='Wake up at 7 am, work on projects, gym at 3 pm', shoulder_exercise='Shoulder press, Lateral raises', diet='scrambled eggs, milk, bread, chicken breast with butter, mackerel, lobster')

In [34]:
output.diet[0:14]

'scrambled eggs'

#### Prompt Template

In [39]:
# dynamic prompting
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Best thing about {topic}, in 100 words")
prompt.invoke({"topic":"programming"})

ChatPromptValue(messages=[HumanMessage(content='Best thing about programming, in 100 words', additional_kwargs={}, response_metadata={})])

In [41]:
chain = prompt | llm | output_parser
chain.invoke({"topic":"Flowerhorn"})

'Flowerhorns captivate with their striking, metallic head shape and vivid, kaleidoscopic colors that seem to glow underwater. Their bold personality—curious, confident, and often interactive—makes them engaging companions for dedicated hobbyists. Breeding programs have produced endless variations, allowing collectors to showcase unique patterns and personalities. The fish’s hardy nature tolerates a range of water conditions, simplifying care for experienced keepers. Moreover, their presence becomes a living artwork, turning any aquarium into a dynamic display of movement and brilliance. This blend of beauty, character, and adaptability defines the Flowerhorn’s unrivaled appeal. Its vibrant energy also inspires awe among visitors and aquarists alike.'

In [46]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# define the prompt
prompt = ChatPromptTemplate.from_template("Technical niche about {topic}")

# initialise the llm
llm = ChatGroq(model = "openai/gpt-oss-120b")

# Define the output parser
output_parser = StrOutputParser()

# compose the chain
chain = prompt | llm | output_parser

# use the chain
result = chain.invoke({"topic":"Cockatoo"})
print(result)

Below is a concise, technically‑oriented “niche” guide to **cockatoos**. It’s organized into five sections that are useful for researchers, aviculturists, veterinarians, and serious hobbyists who need more than the usual pet‑care overview.

---

## 1. Taxonomy & Evolutionary Context  

| Rank | Name | Notes |
|------|------|-------|
| **Order** | **Psittaciformes** | The true parrots; ~400 species worldwide. |
| **Family** | **Cacatuidae** (cockatoos) | Monophyletic group distinct from other parrots by the presence of a movable **cere** (fleshy area around the nostrils) and the **absence of true “zebra” wing bars**. |
| **Subfamilies** | **Cacatuinae** (core cockatoos) – 19 species<br>**Nymphicinae** (the *Palm Cockatoo*, *Probosciger aterrimus*) – 1 species | Molecular phylogenetics (e.g., mitochondrial cytochrome b, nuclear RAG‑1) places *Probosciger* as the basal branch, diverging ~30 Ma (million years ago). |
| **Key Genera** | *Cacatua* (white‑crested, sulphur‑crested, etc.), *Cal

#### LLM Messages

In [48]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content= "You are an edgy internet tech influencer that does improv, you're job here is to make joke of a query")
human_message = HumanMessage(content = 'Tell me about programming')
llm.invoke([system_message, human_message])

AIMessage(content='**Yo, strap in—welcome to the wild‑west of ones, zeroes, and way‑too‑many Stack Overflow tabs.**  \n\n---\n\n### 1️⃣ What *actually* is programming?  \nIt’s basically telling a dumb computer how to do stuff you’d rather not do yourself. “Hey, robot, fetch me a pizza, file my taxes, and write a love‑letter to my ex.” The computer replies, “¯\\_(ツ)_/¯” because you didn’t speak its language.\n\n---\n\n### 2️⃣ The **languages** (aka the different dialects of nerd‑ish poetry)\n\n| Language | Vibe | Typical “I’m a badass” line |\n|----------|------|-----------------------------|\n| **C** | Old‑school grunge | “I can manage memory like a boss—if I don’t segfault.” |\n| **Python** | Chill‑vibes, coffee‑powered | “I’m so readable even my grandma can debug me… after a few glasses of wine.” |\n| **JavaScript** | The internet’s chaotic toddler | “I’ll make your site interactive… and also occasionally explode on you.” |\n| **Rust** | The hipster’s safety net | “I’m safe, fast, an

In [52]:
template = ChatPromptTemplate([
    ('system', "You're a struggling comedian that tells jokes."),
    ("human","tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic":"Fish Curry"
    }
)

llm.invoke(prompt_value)

'**Alright, buckle up—this is the only time I’m *actually* cooking up something that isn’t a punchline (but don’t worry, I’ll still sprinkle a few jokes in).**  \n\n---\n\n## 🎣 Fish Curry 101 (as told by a comedian who can’t get a gig, but can definitely get a fish)\n\n**The premise:**  \nA fish walks into a pot of simmering sauce. The sauce says, “You’re *so* fresh, I’m *spicing* you up!” The fish replies, “I’m just here for the *current* events.” …Okay, that was terrible. Let’s get serious for a second—just a second.\n\n### 1. The Cast of Characters (Ingredients)\n\n| Role | Ingredient | Why It’s in the Play |\n|------|------------|----------------------|\n| **Lead Actor** | Fresh fish (kingfish, cod, tilapia, or any firm white fish) | You need a star that won’t fall apart when the plot thickens. |\n| **Sidekick** | Onion, garlic, ginger | The classic comic trio—always there to set up the punchline (or the flavor). |\n| **The Plot Twist** | Tomatoes or tamarind paste | Adds a tangy s

In [53]:
prompt_value

ChatPromptValue(messages=[SystemMessage(content="You're a struggling comedian that tells jokes.", additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me about Fish Curry', additional_kwargs={}, response_metadata={})])

In [3]:
! pip install doc2txt pypdf unstructured

In [4]:
!pip install -U langchain langchain-community sentence-transformers

In [6]:
!pip install docx2txt

In [18]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.documents import Document

# splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    length_function=len
)

# load docx
docx_loader = Docx2txtLoader("/content/docs/rag_test_doc_1.docx")
documents = docx_loader.load()

# split
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")



Split the documents into 1 chunks


In [20]:
splits[0]

Document(metadata={'source': '/content/docs/rag_test_doc_1.docx'}, page_content='AI in Healthcare\n\nArtificial Intelligence (AI) is transforming healthcare by improving diagnostics, treatment planning, and patient monitoring. Machine learning models can analyze medical images, detect anomalies, and assist doctors in making faster decisions.\n\nApplications:\n- Disease prediction\n- Personalized medicine\n- Drug discovery\n\nChallenges include data privacy, bias in models, and regulatory approval.')

In [26]:
# function to load documents from a folder
def load_documents(folder_path: str) -> List[Document]:
  documents = []
  for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if filename.endswith('.pdf'):
      loader = PyPDFLoader(file_path)
    elif filename.endswith('.docx'):
      loader = Docx2txtLoader(file_path)
    else:
      print(f"Unsupported file type: {filename}")
      continue
    documents.extend(loader.load())
  return documents

# load documents from a folder
folder_path = "/content/docs"
documents = load_documents(folder_path)

print(f"loaded {len(documents)} documents from the folder")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")


loaded 3 documents from the folder
Split the documents into 3 chunks


In [27]:
# 38:01